# GameTheory 16k — Théorie des enchères : le prix mesuré de l'information

**Objectifs** : analyser la vérité, les revenus et la réserve optimale ; comparer deux formats.

**Prérequis** : espérance, intégration et C#. **Durée** : 60 minutes.

**Navigation** : [Version Python](GameTheory-16k-Theorie-des-Encheres-Myerson-Python.ipynb) | [<< 16c-Extraction-de-Revenu-DSIC-IR](GameTheory-16c-Extraction-de-Revenu-DSIC-IR-Python.ipynb) | [Index GameTheory](README.md) | [17-MultiAgent-RL >>](GameTheory-17-MultiAgent-RL-Python.ipynb)

**La théorie des enchères** (Myerson, Vickrey) : un vendeur met aux enchères un bien unique,
$n$ acquéreurs ont chacun une **valeur privée** qu'ils seuls connaissent. On **mesure** le
prix que l'asymétrie d'information impose au mécanisme — le symétrique exact de 16j
(screening / loyer d'information), du côté du vendeur.

À la famille **16 (Mécanismes)**, à côté de 16j : on calcule, pour des valeurs
$Uniforme[0,1]$, (1) la **vrais-facité** de l'enchère d'ordre 2 (Vickrey), (2) le **revenu
espéré du vendeur**, (3) la **réserve optimale de Myerson** et le **saut de revenu** qu'elle
produit, (4) le **théorème d'équivalence de revenu** (ordre 1 ≈ ordre 2).

Toutes les valeurs sont **calculées** dans le carnet (.NET, CPU seul, seedé), pas énoncées.


## 1. Le modèle

$n$ acquéreurs, chacun avec une **valeur privée** $v_i$ i.i.d. tirée de $Uniforme[0,1]$.
Le bien part au plus haut, sous une règle de paiement. Deux notions portent tout le reste :

| | Formule | Rôle |
|---|---|---|
| Fonction de répartition | $F(v) = v$  | probabilité qu'une valeur soit $\le v$ |
| Densité | $f(v) = 1$  | densité uniforme sur $[0,1]$ |
| **Valeur virtuelle** | $\varphi(v) = v - \dfrac{1-F(v)}{f(v)} = 2v - 1$  | le poids *économique* d'un acquéreur de valeur $v$ |

La **valeur virtuelle** est la clé de Myerson : en dessous de $r^*$ (où $\varphi=0$), un
acquéreur a une valeur virtuelle **négative** — le vendre, c'est créer de la rente d'information
qui coûte plus qu'il ne rapporte.


In [1]:
using System;
using System.Linq;
using System.Globalization;

CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
Console.WriteLine($"Runtime : {System.Runtime.InteropServices.RuntimeInformation.FrameworkDescription}");
// Sous U[0,1], la valeur virtuelle est affine ; sa racine donne la réserve candidate 1/2.
// Les expériences suivantes distinguent ce résultat analytique des estimateurs Monte-Carlo.
int n = 3;
double F(double v) => Math.Clamp(v, 0, 1);
double f(double v) => v >= 0 && v <= 1 ? 1 : 0;
double phi(double v) => v - (1 - F(v)) / (f(v) > 0 ? f(v) : 1);
double rStar = 0.5;
Console.WriteLine($"n = {n}, réserve = {rStar:F2}");
foreach (var v in new[]
{
    0.0,
    0.25,
    0.5,
    0.75,
    1.0
}

)
    Console.WriteLine($"v = {v:F2} -> phi = {phi(v):F3}");


Runtime : .NET 10.0.14


n = 3, réserve = 0.50


v = 0.00 -> phi = -1.000


v = 0.25 -> phi = -0.500


v = 0.50 -> phi = 0.000


v = 0.75 -> phi = 0.500


v = 1.00 -> phi = 1.000


### Lecture

$\varphi$ croît de $-1$ (à $v=0$) à $+1$ (à $v=1$) et s'annule à $v=1/2$. Un acquéreur
dont la valeur est **sous** $1/2$ a une valeur virtuelle négative : si on le laisse remporter,
il paie moins que la rente d'information qu'il extrait. La réserve optimale **coupe** juste à
$1/2$ — c'est toute la décision de conception.


### Exercice 1 — Valeur virtuelle pour $F(v)=v^2$

Écrivez une fonction de $v>0$ qui calcule la valeur virtuelle ; puis cherchez son zéro dans $[0,1]$. Renseignez aussi la réserve optimale dans `reserve_beta` (Python) ou `reserveBeta` (C#). Le garde vérifie plusieurs valeurs et la racine annoncée.

In [2]:
Func<double,double?> virtuelleBeta = v => {
    // TODO : valeur virtuelle pour F(v)=v²
    return null;
};
double? reserveBeta = null; // TODO : réserve optimale de cette distribution
var sondesBeta = new[]{0.25,0.5,0.8};
var reponsesBeta = sondesBeta.Select(virtuelleBeta).ToArray();
if(reponsesBeta.Any(x=>x is null) || reserveBeta is null) Console.WriteLine("Exercice 1 à compléter");
else { for(int i=0;i<sondesBeta.Length;i++){double x=reponsesBeta[i].Value; if(!double.IsFinite(x)||Math.Abs(x-new[]{-1.625,-0.25,0.575}[i])>1e-9) throw new Exception("Sonde incorrecte");} double r=reserveBeta.Value; if(!double.IsFinite(r)||r<=0||r>=1) throw new Exception("Réserve hors domaine"); double residuDistribution=r-(1-r*r)/(2*r); if(Math.Abs(residuDistribution)>1e-9) throw new Exception("Réserve incompatible avec F et f"); var racine=virtuelleBeta(r); if(racine is null||!double.IsFinite(racine.Value)||Math.Abs(racine.Value)>1e-9) throw new Exception("Réserve incorrecte"); Console.WriteLine("Exercice 1 : valeur virtuelle et réserve validées"); }

Exercice 1 à compléter


## 2. Vrais-facité de l'enchère d'ordre 2 (Vickrey)

En **enchère d'ordre 2** (au second prix), le gagnant paie la **plus haute offre des autres**
(dite $x$), pas la sienne. Soit $v$ la valeur qu'un acquéreur a réellement pour le bien, et $b$
son **offre**. Il remporte le bien **si et seulement si** $b \ge x$ ; s'il gagne, il paie $x$.
Son surplus est donc :

$$U(v; b, x) = \begin{cases} v - x & \text{si } b \ge x \\ 0 & \text{si } b < x . \end{cases}$$

On **mesure** que révéler sa vraie valeur ($b = v$) maximise ce surplus, **quelle que soit** la
compétition $x$ : l'offre franche est une **stratégie dominante**.


In [3]:
double Surplus(double v, double x, double b) => b >= x ? v-x : 0;
foreach (var (v,x) in new[]{(0.9,0.4),(0.9,0.8),(0.4,0.3),(0.4,0.6),(0.6,0.6)}) {
    double max = Enumerable.Range(0,2001).Max(i => Surplus(v,x,i/2000.0));
    Console.WriteLine($"v={v:F2}, x={x:F2}, vrai={Surplus(v,x,v):F3}, maximum={max:F3}, optimal={Math.Abs(Surplus(v,x,v)-max)<1e-9}");
}

v=0.90, x=0.40, vrai=0.500, maximum=0.500, optimal=True


v=0.90, x=0.80, vrai=0.100, maximum=0.100, optimal=True


v=0.40, x=0.30, vrai=0.100, maximum=0.100, optimal=True


v=0.40, x=0.60, vrai=0.000, maximum=0.000, optimal=True


v=0.60, x=0.60, vrai=0.000, maximum=0.000, optimal=True


### Lecture

Sur chaque ligne, offrir sa vraie valeur $v$ atteint le surplus maximal : si $v > x$ elle gagne
et capte $v-x$ ; si $v < x$ elle perd (surplus $0$), ce qui est de toute façon le meilleur
possible. L'acquéreur **n'a pas besoin de modéliser ses adversaires** pour bien jouer — c'est
la propriété qui rend l'ordre 2 si précieux, et que 16j ne pouvait pas offrir.


## 3. Revenu du vendeur — ordre 2 sans réserve

Quand tous offrent franchement, le vendeur d'un ordre 2 **sans réserve** reçoit la **2ᵉ plus
haute valeur** des $n$ acquéreurs. C'est la **2ᵉ statistique d'ordre** $X_{(n-1)}$. Pour des
valeurs $Uniforme[0,1]$, son espérance a une forme close :

$$\mathbb{E}[X_{(n-1)}] = \frac{n-1}{n+1} .$$

On la **mesure** — formule close d'un côté, Monte-Carlo seedé de l'autre.


In [4]:
double Base(int count) => (count-1.0)/(count+1);
double SimSecond(int count, int trials=200000, int seed=42) {
    var random=new Random(seed); double sum=0;
    for(int t=0;t<trials;t++) { double first=0,second=0; for(int i=0;i<count;i++) { var x=random.NextDouble(); if(x>first){second=first;first=x;} else if(x>second) second=x; } sum+=second; }
    return sum/trials;
}
foreach(int count in new[]{2,3,5,10}) Console.WriteLine($"n={count}: analytique={Base(count):F4}, MC={SimSecond(count):F4} (200000 tirages)");

n=2: analytique=0.3333, MC=0.3336 (200000 tirages)


n=3: analytique=0.5000, MC=0.4998 (200000 tirages)


n=5: analytique=0.6667, MC=0.6661 (200000 tirages)


n=10: analytique=0.8182, MC=0.8178 (200000 tirages)


### Lecture

La formule close et le Monte-Carlo diffèrent au plus de $0.0006$ dans les valeurs affichées avec les tirages .NET : le revenu de référence du
vendeur est **mesuré**, pas supposé. Pour $n=3$ il vaut $0.5000$ — c'est le **benchmark** que
la conception de mécanismes va chercher à dépasser.


## 4. La réserve de Myerson — le saut de revenu

Le **théorème de Myerson** : l'enchère qui maximise le revenu espéré du vendeur est un
**ordre 2 avec réserve** $r^*$, où $r^*$ est la valeur de **valeur virtuelle nulle**
($r^*=1/2$ ici). Règle : on vend **si** le plus haut bid atteint $r^*$, et le gagnant paie
$\max(r^*, \text{2ᵉ bid})$.

On **mesure** son revenu espéré — par intégrale (quadrature du triangle des deux plus hautes
valeurs) **et** par Monte-Carlo — et on le compare au benchmark sans réserve.


In [5]:
// Intégrer sur le deuxième ordre statistique, en distinguant paiement de réserve et deuxième prix.
// Les demi-poids aux deux bornes sont ceux de la règle des trapèzes sur 20 001 points.
double ReserveRevenue(int count, double r)
{
    int intervals = 20000;
    double sum = 0;
    for (int i = 0; i <= intervals; i++)
    {
        double x = (double)i / intervals;
        double term = x >= r ? x * (1 - x) : r * (1 - r);
        double y = term * count * (count - 1) * Math.Pow(x, count - 2);
        sum += (i == 0 || i == intervals ? 0.5 : 1) * y;
    }

    return sum / intervals;
}

// Garder les deux plus grandes offres à chaque tirage ; aucune vente si la première reste sous la réserve.
double SimReserve(int count, double r, int trials = 400000, int seed = 42)
{
    var random = new Random(seed);
    double sum = 0;
    for (int t = 0; t < trials; t++)
    {
        double first = 0, second = 0;
        for (int i = 0; i < count; i++)
        {
            double x = random.NextDouble();
            if (x > first)
            {
                second = first;
                first = x;
            }
            else if (x > second)
                second = x;
        }

        if (first >= r)
            sum += Math.Max(r, second);
    }

    return sum / trials;
}

// Comparer quadrature et simulation au même n sans imposer une identité de leurs arrondis.
n = 3;
double baseline = Base(n), optimal = ReserveRevenue(n, rStar), mc = SimReserve(n, rStar);
Console.WriteLine($"n={n} : sans réserve={baseline:F5}, quadrature={optimal:F5}, MC={mc:F5}, saut={optimal - baseline:F5} (+{100 * (optimal - baseline) / baseline:F2} %)");
if (Math.Abs(optimal - mc) >= 0.005)
    throw new Exception("Divergence Monte-Carlo");


n=3 : sans réserve=0.50000, quadrature=0.53125, MC=0.53138, saut=0.03125 (+6.25 %)


### Lecture — le résultat central

Pour $n=3$, la réserve $1/2$ fait monter le revenu espéré de $0.50000$ à $0.53125$
(**$+6.25\%$**), et l'intégrale et le Monte-Carlo s'accordent. C'est le **prix mesuré de
l'information** du côté vendeur : la réserve réduit la rente attendue de certains acquéreurs, au prix de ventes abandonnées. Modeste pour $n=3$ — et c'est
là que le §6 révèle l'essentiel.


### Exercice 2 — Paiement avec réserve

Écrivez une fonction retournant l’indice du gagnant (à partir de zéro) et le revenu du vendeur ; sans vente, utilisez -1 pour l’indice pour une liste de trois offres et une réserve. Essayez aussi les cas sans vente et où la réserve détermine le prix.

In [6]:
Func<double[],double,(int gagnant,double revenu)?> paiementEnchere = (offres,reserve) => {
    // TODO : indice du gagnant et revenu, ou (-1,0) sans vente
    return null;
};
var casPaiement = new[]{(new[]{0.3,0.8,0.55},0.5,1,0.55),(new[]{0.1,0.2,0.3},0.5,-1,0.0),(new[]{0.1,0.4,0.8},0.5,2,0.5)};
var reponsesPaiement=casPaiement.Select(c=>paiementEnchere(c.Item1,c.Item2)).ToArray();
if(reponsesPaiement.Any(x=>x is null)) Console.WriteLine("Exercice 2 à compléter");
else { for(int i=0;i<casPaiement.Length;i++){var x=reponsesPaiement[i].Value; if(x.gagnant!=casPaiement[i].Item3 || !double.IsFinite(x.revenu)||Math.Abs(x.revenu-casPaiement[i].Item4)>1e-9) throw new Exception("Sonde incorrecte");} Console.WriteLine("Exercice 2 : sondes validées"); }

Exercice 2 à compléter


## 5. Équivalence de revenu — ordre 1 ≈ ordre 2

En **enchère d'ordre 1** (au premier prix, le gagnant paie **son** bid), l'équilibre
symétrique est $b(v) = \frac{n-1}{n}\,v$. Le théorème d'équivalence de revenu dit que,
**en espérance pour le vendeur**, l'ordre 1 et l'ordre 2 (sans réserve) rapportent la même
chose. On le **mesure** : on simule l'ordre 1 à l'équilibre et on compare.


In [7]:
double SimFirst(int count,int trials=300000,int seed=42){var random=new Random(seed); double sum=0; for(int t=0;t<trials;t++){double best=0; for(int i=0;i<count;i++)best=Math.Max(best,random.NextDouble()); sum+=(count-1.0)/count*best;} return sum/trials;}
Console.WriteLine($"n=3 : premier prix MC={SimFirst(3):F4}, second prix analytique={Base(3):F4} (300000 tirages)");

n=3 : premier prix MC=0.4996, second prix analytique=0.5000 (300000 tirages)


### Lecture

L'ordre 1 (où chacun **sous-enchérit** pour ne pas payer trop) et l'ordre 2 (où chacun offre
franchement) drainent le **même** revenu espéré du vendeur. La *forme* de l'enchère change le
comportement, pas le revenu — l'unique bémol étant la réserve, qui est le levier de conception
vraiment efficace (c'est elle qui créait le saut du §4).


## 6. Sensibilité — le saut de réserve en fonction de $n$

La réserve rapporte-t-elle toujours la même chose ? On **mesure** le saut de revenu qu'elle
produit quand $n$ croît.


In [8]:
foreach(int count in new[]{2,3,5,8,12}) { double b=Base(count), o=ReserveRevenue(count,rStar); Console.WriteLine($"n={count,2} sans réserve={b:F4}, avec réserve={o:F4}, saut={o-b:F4} ({100*(o-b)/b:F2} %)"); }

n= 2 sans réserve=0.3333, avec réserve=0.4167, saut=0.0833 (25.00 %)


n= 3 sans réserve=0.5000, avec réserve=0.5312, saut=0.0312 (6.25 %)


n= 5 sans réserve=0.6667, avec réserve=0.6719, saut=0.0052 (0.78 %)


n= 8 sans réserve=0.7778, avec réserve=0.7782, saut=0.0004 (0.06 %)


n=12 sans réserve=0.8462, avec réserve=0.8462, saut=0.0000 (0.00 %)


### Lecture

Le saut de la réserve **fond** quand la compétition s'épaissit : $+25\%$ pour $n=2$,
$+6.25\%$ pour $n=3$, et presque rien pour $n=12$. Avec beaucoup d'acheteurs, la 2ᵉ plus
haute enchère monte déjà très haut et capture la rente ; la réserve n'a presque plus de marge.
**Concevoir une réserve est un levier quand le marché est mince, pas quand il est saturé.**


### Exercice 3 — Seuil de compétition

Écrivez une fonction qui trouve le plus petit $n\ge2$ dont le saut relatif de réserve est strictement inférieur au seuil donné. Explorez notamment 1 %.

In [9]:
Func<double,int?> seuilCompetition = seuil => {
    // TODO : parcourir les nombres d'acquéreurs
    return null;
};
var seuils = new[]{0.1,0.01,0.001};
var reponsesSeuil=seuils.Select(seuilCompetition).ToArray();
if(reponsesSeuil.Any(x=>x is null)) Console.WriteLine("Exercice 3 à compléter");
else { for(int i=0;i<seuils.Length;i++){int k=reponsesSeuil[i].Value; if(k<2||k>30) throw new Exception("Hors plage"); Func<int,double> saut=j=>(ReserveRevenue(j,0.5)-Base(j))/Base(j); if(saut(k)>=seuils[i] || Enumerable.Range(2,k-2).Any(j=>saut(j)<seuils[i])) throw new Exception("Seuil incorrect");} Console.WriteLine("Exercice 3 : sondes validées"); }

Exercice 3 à compléter


## 7. Exercices — récapitulatif

Les trois exercices répartis ci-dessus portent sur la valeur virtuelle, le paiement et le seuil de compétition. Comparez vos résultats aux lectures des sections 1, 4 et 6.

## À retenir

- **L'ordre 2 est vrai par construction** : révéler sa valeur est une stratégie dominante,
  mesurée ligne par ligne — l'acquéreur n'a pas à deviner ses rivaux.
- **L'asymétrie a un prix du côté vendeur** : la réserve de Myerson ($r^*=1/2$ ici) enlève la
  rente d'information et fait monter le revenu espéré — **mesuré** ici à $+6.25\%$ pour $n=3$.
- **Ce levier s'éteint avec la compétition** : le saut de la réserve fond quand $n$ croît
  ($+25\%$ à $n=2$, quasi nul à $n=12$) — la conception de la réserve est un art de
  **marché mince**.
- **La forme ne change pas le revenu** : ordre 1 et ordre 2 (sans réserve) sont équivalents en
  revenu espéré ; une réserve optimale peut augmenter ce revenu.

C'est le miroir vendeur de 16j : là où 16j mesurait le **loyer d'information** que le
principal paie à l'agent informé, 16k mesure la **réserve** que le vendeur impose pour ne pas
payer cette rente — même théorème (Myerson), deux faces du même mécanisme.

**Trois mesures à retenir.**

- **La vrais-facité de l'ordre 2 est mesurée ligne par ligne** : révéler sa valeur est une stratégie dominante, et l'acquéreur n'a pas à deviner ses rivaux.
- **Le saut de réserve** : la réserve de Myerson `r* = 1/2` fait monter le revenu espéré de `+25.00 %` pour `n = 2`, de `+6.25 %` pour `n = 3`, de `+0.78 %` pour `n = 5`, de `+0.06 %` pour `n = 8`, et de `+0.00 %` pour `n = 12`.
- **L'équivalence de revenu** : à réserve nulle, l'ordre 1 et l'ordre 2 ont le même revenu espéré (`0.3333` pour `n = 2`, `0.5000` pour `n = 3`, `0.6667` pour `n = 5`, `0.7778` pour `n = 8`) ; une réserve optimale peut augmenter ce revenu.

**Réserve de portée.** Les valeurs sont calculées pour des valeurs **Uniforme[0, 1]** et une offre symétrique : le `r* = 1/2` mesuré ici est celui de cette distribution, pas une constante. L'exercice 1 (distribution `F(v) = v²`) déplace la réserve optimale — c'est le sens du titre du carnet, « le prix mesuré de l'information ».


## Vérifiez votre compréhension

1. L'ordre 2 (Vickrey) est vrai par construction : l'acquéreur paie le prix de la seconde meilleure offre. Le vendeur y laisse-t-il quelque chose ?

   <details><summary>Réponse</summary>

   Oui : sans réserve, le vendeur renonce au gain de revenu que peut apporter une réserve optimale. À réserve nulle, l'ordre 1 et l'ordre 2 ont le **même** revenu espéré (`0.3333` pour `n = 2`, `0.5000` pour `n = 3`, `0.6667` pour `n = 5`, `0.7778` pour `n = 8`, `0.8462` pour `n = 12`). C'est le théorème d'équivalence de revenu, vérifié numériquement. Dans ce modèle symétrique les deux formats ont le même revenu et la même allocation en espérance ; leurs offres et paiements réalisés diffèrent. L'ordre 2 ne demande pas de connaître les stratégies adverses, contrairement au calcul de l'équilibre de premier prix.

   </details>

2. Le saut de réserve tombe à `+0.00 %` pour `n = 12`. La réserve ne sert donc plus à rien ?

   <details><summary>Réponse</summary>

   Le gain reste positif pour tout $n$ fini mais devient très petit : $+0.00\%$ est un arrondi. Quand `n` acquéreurs sont en lice, l'ordre 2 sans réserve extrait déjà presque tout le surplus : la rente d'information résiduelle est négligeable, et le seuil de compétition (exercice 3) est précisément l'endroit où le saut passe sous 1 %. La portée du résultat est donc conditionnelle : la réserve est un levier de **marché mince**, et son intérêt disparaît quand la concurrence s'épaissit.

   </details>

3. Le revenu espéré croît avec `n` (`0.3333` pour `n = 2` contre `0.8462` pour `n = 12`). Pourquoi le saut de réserve fond-il en même temps ?

   <details><summary>Réponse</summary>

   Les deux mouvements sont de même sens mais de nature différente. Avec `n` acquéreurs, la seconde meilleure offre se rapproche du maximum (ordre des statistiques) : le vendeur capte une part croissante du surplus, d'où la croissance du revenu. Mais la rente qu'il laisse subsister — la différence entre la valeur du bien et ce que paie le gagnant — devient une part **décroissante** de ce surplus total : la réserve a de moins en moins de matière à retirer. D'où un saut qui s'éteint alors même que le revenu, lui, continue de monter.

   </details>

4. La réserve de Myerson vaut `1/2` ici. Est-ce une constante ?

   <details><summary>Réponse</summary>

   Non : c'est la réserve optimale **pour la distribution posée au §1** (valeurs Uniforme[0, 1], offre symétrique). Le procédé de Myerson ne donne pas un nombre, il donne une règle : la réserve est le point où la **valeur virtuelle** s'annule, et cette valeur virtuelle dépend de la distribution des valeurs. L'exercice 1 (distribution `F(v) = v²`) déplace donc la réserve optimale — et c'est précisément le prix « mesuré » de l'information, non une constante à recopier.

   </details>

## Pour aller plus loin

**Carnet suivant.** [GameTheory-17 — MultiAgent RL](GameTheory-17-MultiAgent-RL-Python.ipynb) -- la suite de la famille : là où ce carnet mesure le prix d'un mécanisme d'enchère statique, le 17 pose la question de l'**apprentissage** des agents dans un jeu répété.

**Lectures.**

- Les références complètes du carnet voisin `GameTheory-16c` listent **Myerson et Satterthwaite (1983)** (*Efficient mechanisms for bilateral trade*) et **Rochet (1987)** (rationalisabilité en contexte quasi-linéaire) — l'appareil théorique du prix de réserve.
- Le mécanisme d'enchère de référence est celui du carnet owner : l'enchère de second prix de Vickrey, dont l'équivalence de revenu avec l'enchère de premier prix est l'un des résultats établis de la famille.

**Carnets voisins.**

- [GameTheory-16j — Screening et loyer d'information](GameTheory-16j-Screening-Loyer-Information-Python.ipynb) -- le miroir : le loyer d'information côté principal, mesuré par le même type de décomposition.
- [GameTheory-16c — Extraction de revenu](GameTheory-16c-Extraction-de-Revenu-DSIC-IR-Python.ipynb) -- le designer qui maximise le revenu, et l'écart mesuré entre l'espace qu'il visite et l'espace complet.
- [GameTheory-16 — Mécanismes](GameTheory-16-MechanismDesign-Python.ipynb) -- le carnet owner : enchères de Vickrey et mécanisme VCG, dont l'équivalence de revenu est une conséquence.
